<a href="https://colab.research.google.com/github/Young-yrx/guizhou-landslide/blob/main/guizhou.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

XGBoost

In [1]:
"""
贵州水城滑坡易发性评价 —— 基于 XGBoost 的二分类模型与制图
要求：已准备好六个影响因子的栅格数据（与研究区范围一致、分辨率相同）
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 机器学习库
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import xgboost as xgb

# 栅格数据处理库
import rasterio
from rasterio.transform import Affine
from rasterio.crs import CRS

# 忽略警告（可选）
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# -----------------------------------
# 1. 读取并合并样本数据
# -----------------------------------
# 读取滑坡点数据（注意列顺序）
df_landslide = pd.read_excel('landslide_TableToExcel - 副本.xlsx', sheet_name='landslide')
# 选取特征列，并按统一顺序排列（Curvatu, Aspect, Slope, Reclass, PTC, TWI）
feature_cols = ['Curvatu_ASTG1', 'Aspect_ASTGT1', 'Slope_ASTGTM1', 'Reclass_ASTG1', 'PTC', 'TWI']
X_landslide = df_landslide[feature_cols].copy()
y_landslide = np.ones(len(X_landslide))  # 滑坡标签 = 1

# 读取非滑坡点数据（注意列顺序不同，需调整）
df_non = pd.read_excel('non_landslide_TableToExcel - 副本.xlsx', sheet_name='non_landslide')
# 非滑坡文件中列名为：Reclass_ASTG1, Curvatu_ASTG1, Aspect_ASTGT1, Slope_ASTGTM1, TWI, PTC
# 需要按统一顺序提取
X_non = df_non[['Curvatu_ASTG1', 'Aspect_ASTGT1', 'Slope_ASTGTM1', 'Reclass_ASTG1', 'PTC', 'TWI']].copy()
y_non = np.zeros(len(X_non))  # 非滑坡标签 = 0

# 合并数据
X = pd.concat([X_landslide, X_non], axis=0, ignore_index=True)
y = np.concatenate([y_landslide, y_non])

print(f"总样本数: {len(X)}，其中滑坡样本: {int(y.sum())}，非滑坡样本: {len(y) - int(y.sum())}")

In [ ]:
# -----------------------------------
# 2. 划分训练集与测试集
# -----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # 分层抽样，保持类别比例
)

print(f"训练集样本数: {len(X_train)}，测试集样本数: {len(X_test)}")

In [ ]:
# -----------------------------------
# 3. 训练 XGBoost 分类模型
# -----------------------------------
# 计算类别权重，处理样本不平衡（滑坡/非滑坡比例约1:1，但可加强少数类权重）
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

xgb_model = xgb.XGBClassifier(
    n_estimators=200,          # 树的数量
    max_depth=5,               # 树的最大深度
    learning_rate=0.05,        # 学习率
    subsample=0.8,             # 每棵树使用的样本比例
    colsample_bytree=0.8,      # 每棵树使用的特征比例
    scale_pos_weight=scale_pos_weight,  # 正负样本权重平衡
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

In [ ]:
# -----------------------------------
# 4. 模型评估
# -----------------------------------
y_pred = xgb_model.predict(X_test)
y_pred_prob = xgb_model.predict_proba(X_test)[:, 1]  # 获取正类（滑坡）概率

# 准确率
acc = accuracy_score(y_test, y_pred)
# ROC-AUC
auc = roc_auc_score(y_test, y_pred_prob)

print(f"测试集准确率: {acc:.4f}")
print(f"测试集 AUC: {auc:.4f}")
print("\n分类报告:")
print(classification_report(y_test, y_pred, target_names=['非滑坡', '滑坡']))

# 绘制混淆矩阵
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['非滑坡', '滑坡'], yticklabels=['非滑坡', '滑坡'])
plt.xlabel('预测值')
plt.ylabel('真实值')
plt.title('混淆矩阵')
plt.show()

# 特征重要性
importance = xgb_model.feature_importances_
feat_imp = pd.Series(importance, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8,5))
feat_imp.plot(kind='bar')
plt.title('XGBoost 特征重要性')
plt.ylabel('重要性得分')
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------
# 5. 对研究区栅格数据进行预测并导出易发性图
# -----------------------------------
"""
注意：本部分需要您预先准备好六个影响因子的栅格文件，且满足以下要求：
- 所有栅格具有相同的地理范围、像元大小和坐标系
- 栅格文件名建议与特征名对应，例如：
  curvature.tif, aspect.tif, slope.tif, reclass.tif, ptc.tif, twi.tif
- 读取顺序必须与训练时的特征顺序完全一致
"""

# 请根据实际文件路径修改
raster_files = [
    'curvature.tif',   # 曲率
    'aspect.tif',      # 坡向
    'slope.tif',       # 坡度
    'reclass.tif',     # 重分类（可能是地层或岩性）
    'ptc.tif',         # PTC
    'twi.tif'          # TWI
]

# 打开第一个栅格获取元数据（地理变换、坐标系、尺寸等）
with rasterio.open(raster_files[0]) as src:
    profile = src.profile
    height = src.height
    width = src.width
    transform = src.transform
    crs = src.crs

# 创建一个数组存储所有特征（高度, 宽度, 波段数）
feature_stack = np.zeros((height, width, len(raster_files)), dtype=np.float32)

# 读取所有栅格并堆叠
for i, fpath in enumerate(raster_files):
    with rasterio.open(fpath) as src:
        feature_stack[:, :, i] = src.read(1)  # 读取第一波段

# 处理 NoData 值（假设无效值为 -9999 或 NaN，根据实际情况调整）
# 这里将无效值设为 NaN，以便后续生成掩膜
feature_stack[feature_stack < -9990] = np.nan

# 将三维数组转换为二维表格（像素数, 特征数），并剔除含有 NaN 的像素
rows, cols, bands = feature_stack.shape
feature_2d = feature_stack.reshape(-1, bands)
valid_mask = ~np.isnan(feature_2d).any(axis=1)  # 所有特征均有效的像素
valid_features = feature_2d[valid_mask]

print(f"总像素数: {rows*cols}，有效像素数: {valid_features.shape[0]}")

# 使用模型预测滑坡概率（只预测有效像素）
if valid_features.shape[0] > 0:
    pred_prob = xgb_model.predict_proba(valid_features)[:, 1]
else:
    raise ValueError("没有有效像素，请检查栅格数据范围和 NoData 值设置。")

# 创建全图概率数组，默认填充 NaN
prob_map = np.full(rows * cols, np.nan, dtype=np.float32)
prob_map[valid_mask] = pred_prob
prob_map = prob_map.reshape(rows, cols)

In [ ]:
# -----------------------------------
# 6. 保存滑坡易发性图（GeoTIFF）
# -----------------------------------
output_file = 'landslide_susceptibility_xgboost.tif'

# 更新元数据：单波段，数据类型为 float32，压缩方式可选
profile.update(
    dtype=rasterio.float32,
    count=1,
    compress='lzw',
    nodata=np.nan
)

with rasterio.open(output_file, 'w', **profile) as dst:
    dst.write(prob_map, 1)

print(f"滑坡易发性图已保存至: {output_file}")

In [ ]:
# -----------------------------------
# 7. 可选：绘制易发性图预览
# -----------------------------------
plt.figure(figsize=(10, 8))
plt.imshow(prob_map, cmap='RdYlGn_r', vmin=0, vmax=1)
plt.colorbar(label='滑坡发生概率')
plt.title('贵州水城滑坡易发性图 (XGBoost)')
plt.axis('off')
plt.show()